# Paper v2 — from weight disparity to reservoir memory

Analyses and figures of the v2 manuscript and its Supplementary Information.

| § | step | needs | writes |
|---|---|---|---|
| 1 | analytical chain: ESN memory at input scales 1 and $10^{-5}$, symmetric surrogates (`src/chain_analysis.py` + the C++ engine) | `procdata/data_with_metrics.pkl` | `results/chain/` |
| 2–4 | per-subject theory (stable rank, input Gramian, closed-form memory), statistics, every number quoted in the text | the pickle + `results/chain/` | `results/paper/data/` |
| 5 | main figures 2, 4, 5 and SI figures | `results/paper/data/` only | `results/paper/figures/` |
| 6 | noise-free memory in exact (ball) arithmetic, SI Note 2.6 (optional) | the pickle | `results/paper/data/noise_free_exact.csv` |

`procdata/data_with_metrics.pkl` comes from `01_data_and_structure.ipynb`. Figures 1 (atlas and ages)
and 3 (memory paradigm) of the manuscript come from notebooks 01 and 02.

The figures (§5) only read `results/paper/data/`, so they can be redrawn without the connectomes.

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullFormatter
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "engine" / "esn.hpp").exists())
sys.path.insert(0, str(ROOT / "src"))
import chain_analysis as ca  # loaders, ESN runner, statistics helpers

# ---- inputs and outputs --------------------------------------------------------------
DATA_PKL  = ROOT / "procdata" / "data_with_metrics.pkl"   # from 01_data_and_structure.ipynb
CHAIN_DIR = ROOT / "results" / "chain"                    # outputs of the analytical chain (§1)
DATA      = ROOT / "results" / "paper" / "data"           # everything the figures and the text need
OUT       = ROOT / "results" / "paper" / "figures"

# ---- what to run ---------------------------------------------------------------------
RUN_CHAIN    = False   # §1: ESN runs at scales 1 and 1e-5 + surrogates (~10 min on 4 cores); else reuse CHAIN_DIR
RUN_ANALYSIS = True    # §2-4: per-subject quantities and numbers.json (~30 s); skipped if inputs are missing
RIDGE_SWEEP  = False   # §4: also simulate the tanh ESN at four ridge penalties on 300 subjects (~5 min)
RUN_EXACT    = False   # §6: noise-free memory in ball arithmetic (pip install python-flint, ~10 min)
THREADS      = 0       # OpenMP threads for the C++ engine (0 = all cores)
EIGEN        = None    # path to Eigen if it is not found automatically
CHAIN_ARGS   = []      # extra options for src/chain_analysis.py, e.g. ["--surr-real", "5"]

DATA.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
print("repository:", ROOT)

## 1 · Analytical chain

`src/chain_analysis.py` runs the C++ engine (`engine/esn_mc`, `engine/esn_surrogates`, built
automatically) with the manuscript hyper-parameters: ρ = 0.99, ridge = 1e-6 (× input_scale²),
train 0.7, washout 1000, 10 000 steps, τ_max = 20, one input projection per subject, seed 42, at
input scales 1 and 1e-5. The surrogates (Uniform, Broken Stick, Reshuffle, Sign Flip, plus the real
weights made asymmetric as a control) run on a 100-subject subset.

On a Mac the compiler needs OpenMP: set `os.environ["CXX"] = "g++-14"` (Homebrew GCC) before
running, and if the binaries complain about libomp, `DYLD_LIBRARY_PATH=$CONDA_PREFIX/lib`.

In [ ]:
def run_chain():
    cmd = [sys.executable, str(ROOT / "src" / "chain_analysis.py"), "--data", str(DATA_PKL),
           "--out", str(CHAIN_DIR), "--scales", "1", "1e-5", "--threads", str(THREADS), *CHAIN_ARGS]
    if EIGEN: cmd += ["--eigen", str(EIGEN)]
    print(" ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout: print(line, end="")
    if proc.wait() != 0: raise RuntimeError("chain_analysis.py failed")


if RUN_CHAIN:
    run_chain()
elif (CHAIN_DIR / "chain_subject.csv").exists():
    print("reusing", CHAIN_DIR)
else:
    print(f"no chain outputs in {CHAIN_DIR}: set RUN_CHAIN = True to compute them")

## 2 · Closed-form theory (SI Note 2)

Linear reservoir $x(t) = \tilde W x(t-1) + w\,u(t)$, $u$ iid with variance $\sigma_u^2 = 1/3$,
$\tilde W = 0.99\,W/\rho(W)$:

$$C = \sigma_u^2 \sum_j \tilde W^j w w^\top \tilde W^{\top j} \quad\text{(input controllability Gramian)},\qquad
g_\tau = \sigma_u^2\, \tilde W^\tau w,$$

$$m(\tau) = \frac{(g_\tau^\top M g_\tau)^2}{\sigma_u^2\, g_\tau^\top M C M g_\tau},\quad M = (C + \eta I)^{-1},\quad
\eta = \text{ridge}/n_\text{train},\qquad \sum_{\tau\ge0} \frac{g_\tau^\top M g_\tau}{\sigma_u^2} = \sum_k \frac{c_k}{c_k+\eta}\ \text{(effective rank)}.$$

The C++ engine (`engine/esn.hpp`, `mc_linear_theory`) computes the same $m(\tau)$ with its own input
vector; here $w$ is drawn with numpy (seed = subject id), so `MC_theory_py` is an independent
realisation of the input mask.

In [ ]:
RHO, S2U, TAU = 0.99, 1.0 / 3.0, 20
STEPS, WASHOUT, TRAIN = 10000, 1000, 0.7
NTR = min(max(int(round(TRAIN * (STEPS - WASHOUT))), 10), STEPS - WASHOUT - 10)   # as in esn.hpp
RIDGE = 1e-6
ETA0 = RIDGE / NTR                     # noise floor of the manuscript (input scale cancels)
ETA_REL = [1e4, 1e3, 1e2, 1e1, 1.0, 1e-1, 1e-2, 1e-3, 1e-4]
T_SERIES = [1, 2, 3, 5, 10, 20]
AGE_SPLIT = 32.0
MC = "MC_s1e-05"
DS_NAMES = {1: "dHCP", 2: "BCP", 3: "CALM", 4: "RED", 5: "ACE", 6: "HCPd", 7: "HCPya", 8: "camCAN", 9: "HCPa"}

Per-subject quantities: stable rank, local/global heterogeneity factors, truncated Gramian series, input-Gramian spectrum (effective rank, decay rate) and the closed-form memory with its noise-floor sweep.

In [ ]:
def input_gramian(Wn, w):
    """C = s2u sum_j Wn^j w w' Wn'^j by doubling (2^16 terms; |lambda| <= 0.99)."""
    C, A = S2U * np.outer(w, w), Wn.copy()
    for _ in range(16):
        C = C + A @ C @ A.T
        A = A @ A
    return 0.5 * (C + C.T)


def theory_curves(Wn, w, c, U, etas):
    """Closed-form m(tau), tau = 1..TAU, for each eta, from the eigendecomposition of C."""
    g = S2U * w
    P = np.empty((TAU, len(c)))
    for t in range(TAU):
        g = Wn @ g
        P[t] = (U.T @ g) ** 2                         # squared projections of g_tau on C's eigenvectors
    out = []
    for eta in etas:
        num = (P / (c + eta)).sum(1)                  # g' M g
        den = S2U * (P * c / (c + eta) ** 2).sum(1)   # s2u g' M C M g
        out.append(num ** 2 / den)
    return np.array(out)


def decay_rate(c, k_max):
    """Geometric decay ratio q of the leading input-Gramian eigenvalues (log-linear fit)."""
    k = np.arange(min(max(k_max, 3), len(c)))
    slope = np.polyfit(k, np.log(c[k]), 1)[0]
    return float(np.exp(slope))


def subject_quantities(A, seed):
    N = A.shape[0]
    lam_raw, V = np.linalg.eigh(A)
    i = int(np.argmax(np.abs(lam_raw)))
    rho = abs(lam_raw[i]); v1 = V[:, i]
    Wn, lam = A * (RHO / rho), lam_raw * (RHO / rho)
    s = A.sum(1)
    lam2 = lam ** 2
    q = dict(stable_rank=float((A ** 2).sum() / rho ** 2),
             s2t=float(RHO ** 2 * (s ** 2).mean() / rho ** 2),       # global factor <s~^2>
             rho_over_srms=float(rho / np.sqrt((s ** 2).mean())),
             ipr_v1=float((v1 ** 4).sum()), cv_s=float(s.std() / s.mean()),
             m3_py=float((lam ** 3).mean()))
    for T in T_SERIES:
        q[f"gram_T{T}"] = float((lam2 * (1 - lam2 ** T) / (1 - lam2)).mean())
    q["gram_Tinf"] = float((lam2 / (1 - lam2)).mean())                 # = <(G_I)_ii> - 1
    q["n_slow"] = int((np.abs(lam) > 0.9).sum())

    w = np.random.default_rng(int(seed)).uniform(-1.0, 1.0, N)
    C = input_gramian(Wn, w)
    c, U = np.linalg.eigh(C)
    c, U = np.clip(c[::-1], 0.0, None), U[:, ::-1]
    curves = theory_curves(Wn, w, c, U, [ETA0 * e for e in ETA_REL])
    j0 = ETA_REL.index(1.0)
    q.update(MC_theory_py=float(curves[j0].sum()), eff_rank=float((c / (c + ETA0)).sum()),
             n_above=int((c > ETA0).sum()), c1_over_eta=float(c[0] / ETA0))
    q["q_decay"] = decay_rate(c, q["n_above"])
    q["mc_formula"] = float(1 + np.log(q["c1_over_eta"]) / np.log(1 / q["q_decay"]))
    sweep = [dict(eta_rel=e, MC_theory=float(curves[j].sum()),
                  eff_rank=float((c / (c + ETA0 * e)).sum())) for j, e in enumerate(ETA_REL)]
    return q, sweep, c, curves[j0]

Node-level sample (150 subjects): degree, disparity and the walk-expansion terms of the subgraph centrality.

In [ ]:
def node_sample(A):
    N = A.shape[0]
    lam_raw, V = np.linalg.eigh(A)
    rho = np.abs(lam_raw).max()
    Wn, lam = A * (RHO / rho), lam_raw * (RHO / rho)
    k = (A > 0).sum(1).astype(float); s = A.sum(1)
    with np.errstate(divide="ignore", invalid="ignore"):
        Y = np.where(s > 0, (A ** 2).sum(1) / s ** 2, np.nan)
    W2 = Wn @ Wn
    return pd.DataFrame(dict(k=k, Y=Y, Ups=k * Y, Ups_null=2 * k / (k + 1),
                             Cii=(V ** 2) @ np.exp(lam), T2=0.5 * np.diag(W2),
                             T3=np.einsum("ij,ji->i", W2, Wn) / 6.0, s_tilde=Wn.sum(1)))

## 3 · Statistics and the numbers quoted in the text

`numbers()` collects every number of the manuscript and SI into one dictionary, saved as
`results/paper/data/numbers.json`.

In [ ]:
def r_(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    return float(stats.pearsonr(x[m], y[m])[0])


def pcor(d, a, b, z):
    """Partial correlation r(a, b | z), z a column name or list of names."""
    z = [z] if isinstance(z, str) else list(z)
    return float(ca.partial_r(d[a].values, d[b].values, d[z].values)["r"])


def r2(d, y, cols):
    return float(ca.r2_ols(d, y, cols))


def smoothed_min_age(d, col):
    ab = np.floor(d["age"]).astype(int)
    g = d.groupby(ab)[col].mean(); n = d.groupby(ab)[col].size(); g = g[n >= 5]
    sm = g.rolling(5, center=True, min_periods=3).mean()
    a = sm[(sm.index >= 10) & (sm.index <= 60)]
    return int(a.idxmin())


def quad_vertex(age, y, n_boot=2000, seed=0):
    z = (age - age.mean()) / age.std()
    A = np.column_stack([np.ones(len(z)), z, z ** 2])
    b, *_ = np.linalg.lstsq(A, y, rcond=None)
    e = y - A @ b; s2 = (e ** 2).sum() / (len(y) - 3); cov = s2 * np.linalg.inv(A.T @ A)
    t = b[2] / np.sqrt(cov[2, 2]); p = 2 * stats.t.sf(abs(t), len(y) - 3)
    p2 = np.polyfit(age, y, 2); vtx = -p2[1] / (2 * p2[0])
    rng = np.random.default_rng(seed); vs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(age), len(age)); pb = np.polyfit(age[i], y[i], 2)
        vs.append(-pb[1] / (2 * pb[0]) if pb[0] > 0 else np.nan)
    vs = np.array(vs)
    return dict(vertex=float(vtx), ci=[float(np.nanpercentile(vs, 2.5)), float(np.nanpercentile(vs, 97.5))],
                p_quadratic=float(p), coef=[float(x) for x in p2])

In [ ]:
def numbers(df, sweep, nodes, surr, ridge):
    N = {}
    d = df.dropna(subset=[MC]).copy()
    d["log_rs"] = np.log(d["stable_rank"])
    dev, agi = d[d["age"] <= AGE_SPLIT], d[d["age"] > AGE_SPLIT]
    N["n"] = int(len(d))
    N["MC"] = dict(min=float(d[MC].min()), p5=float(d[MC].quantile(.05)), median=float(d[MC].median()),
                   p95=float(d[MC].quantile(.95)), max=float(d[MC].max()), mean=float(d[MC].mean()))
    N["MC_scale1"] = dict(min=float(d["MC_s1"].min()), median=float(d["MC_s1"].median()), max=float(d["MC_s1"].max()))
    N["theory"] = dict(r_sim_cpp=r_(d[MC].values, d["MClin_s1e-05"].values),
                       r_sim_py=r_(d[MC].values, d["MC_theory_py"].values),
                       r_sim_cpp_scale1=r_(d["MC_s1"].values, d["MClin_s1"].values),
                       mean_sim=float(d[MC].mean()), mean_cpp=float(d["MClin_s1e-05"].mean()),
                       mean_py=float(d["MC_theory_py"].mean()))
    N["stable_rank"] = dict(median=float(d["stable_rank"].median()), min=float(d["stable_rank"].min()),
                            max=float(d["stable_rank"].max()),
                            r_MC={k: r_(x["stable_rank"].values, x[MC].values) for k, x in [("all", d), ("dev", dev), ("aging", agi)]},
                            R2_lin=r2(d, MC, ["stable_rank"]), R2_log=r2(d, MC, ["log_rs"]),
                            R2_log_plus_Ginf=r2(d, MC, ["log_rs", "gram_Tinf"]),
                            r_Cii=r_(d["Cii_mean"].values, d["stable_rank"].values),
                            r_Ginf=r_(d["gram_Tinf"].values, d["stable_rank"].values),
                            identity_max_relerr=float(d["identity_relerr"].max()) if "identity_relerr" in d else None)
    N["effective_rank"] = dict(mean=float(d["eff_rank"].mean()), min=float(d["eff_rank"].min()), max=float(d["eff_rank"].max()),
                               r_MC=r_(d["eff_rank"].values, d[MC].values), r_rs=r_(d["eff_rank"].values, d["stable_rank"].values),
                               n_above_mean=float(d["n_above"].mean()), r_MCpy=r_(d["eff_rank"].values, d["MC_theory_py"].values),
                               q_median=float(d["q_decay"].median()),
                               r_q_rs=r_(d["q_decay"].values, d["stable_rank"].values),
                               r_formula_effrank=r_(d["mc_formula"].values, d["eff_rank"].values))
    N["gram_series_r_MC"] = {f"T{T}": r_(d[f"gram_T{T}"].values, d[MC].values) for T in T_SERIES}
    N["gram_series_r_MC"]["Tinf"] = r_(d["gram_Tinf"].values, d[MC].values)
    N["n_slow_r_MC"] = r_(d["n_slow"].values.astype(float), d[MC].values)
    preds = {"stable_rank": "stable_rank", "Cii": "Cii_mean", "abs_lambda": "abs_lam_mean",
             "communicability": "comm_mean", "m3": "m3", "clustering_Cw": "C_w", "disparity_ratio": "Ratio",
             "strength_raw": "strength", "degree": "degree", "avg_controllability": "gram_Tinf",
             "global_factor": "s2t", "density": "density"}
    race = {}
    for name, x in [("all", d), ("dev", dev), ("aging", agi)]:
        race[name] = {}
        for lab, c in preds.items():
            if c not in x: continue
            race[name][lab] = dict(r=r_(x[c].values, x[MC].values),
                                   partial_given_rs=None if c == "stable_rank" else pcor(x, c, MC, "stable_rank"))
    N["predictors"] = race
    N["global_factor"] = dict(r_rho_over_srms=r_(d["s2t"].values, d["rho_over_srms"].values),
                              r_cv_s=r_(d["s2t"].values, d["cv_s"].values),
                              r_ipr=r_(d["s2t"].values, d["ipr_v1"].values),
                              max=float(d["s2t"].max()))
    ages = {}
    for name, x in [("dev", dev), ("aging", agi)]:
        ages[name] = {c: r_(x[c].values, x["age"].values) for c in
                      [MC, "stable_rank", "Ratio", "s2t", "strength", "density", "Cii_mean"] if c in x}
        ages[name]["MC_vs_strength"] = r_(x["strength"].values, x[MC].values)
        ages[name]["partial_MC_age_given_density"] = pcor(x, MC, "age", "density")
    N["age_correlations"] = ages
    N["smoothed_min_age"] = {c: smoothed_min_age(d, c) for c in [MC, "stable_rank", "Cii_mean", "s2t"]}
    coh = {}
    for code, x in d.groupby("dataset"):
        nm = DS_NAMES.get(int(code), str(code))
        e = dict(n=int(len(x)), age_min=float(x["age"].min()), age_max=float(x["age"].max()),
                 MC_mean=float(x[MC].mean()), rs_mean=float(x["stable_rank"].mean()),
                 R2_log_rs=r2(x, MC, ["log_rs"]),
                 partial_ratio_given_global=pcor(x, "Ratio", MC, "s2t"),
                 partial_global_given_ratio=pcor(x, "s2t", MC, "Ratio"))
        if x["age"].std() > 0:
            e.update(r_MC_age=r_(x[MC].values, x["age"].values), r_rs_age=r_(x["stable_rank"].values, x["age"].values),
                     r_ratio_age=r_(x["Ratio"].values, x["age"].values), r_global_age=r_(x["s2t"].values, x["age"].values))
        if "sex" in x and x["sex"].nunique() == 2:
            a, b = x[x["sex"] == 0][MC], x[x["sex"] == 1][MC]
            sp = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
            e.update(sex_n0=int(len(a)), sex_n1=int(len(b)), sex_d=float((b.mean() - a.mean()) / sp),
                     sex_p=float(stats.mannwhitneyu(a, b).pvalue))
        coh[nm] = e
    N["cohorts"] = coh
    cc = d[d["dataset"] == 8]
    if len(cc) > 50:
        bins = [18, 25, 32, 40, 50, 60, 70, 80, 90]
        g = cc.groupby(pd.cut(cc["age"], bins), observed=True)
        N["camcan"] = dict(bins=[f"{int(i.left)}-{int(i.right)}" for i in g.groups.keys()],
                           MC=[float(v) for v in g[MC].mean()], rs=[float(v) for v in g["stable_rank"].mean()],
                           ratio=[float(v) for v in g["Ratio"].mean()], s2t=[float(v) for v in g["s2t"].mean()],
                           n=[int(v) for v in g.size()],
                           MC_quad=quad_vertex(cc["age"].values, cc[MC].values),
                           rs_quad=quad_vertex(cc["age"].values, cc["stable_rank"].values))
        lo, hi = cc[cc["age"] <= 40], cc[cc["age"] >= 35]
        N["camcan"].update(r_le40=r_(lo[MC].values, lo["age"].values), p_le40=float(stats.pearsonr(lo[MC], lo["age"])[1]),
                           r_ge35=r_(hi[MC].values, hi["age"].values), p_ge35=float(stats.pearsonr(hi[MC], hi["age"])[1]))
    def overlap(a, b, lo, hi):
        A = d[(d["dataset"] == a) & d["age"].between(lo, hi)]; B = d[(d["dataset"] == b) & d["age"].between(lo, hi)]
        out = {}
        for c in [MC, "stable_rank", "Ratio", "s2t"]:
            out[c] = dict(a=float(A[c].mean()), b=float(B[c].mean()), na=int(len(A)), nb=int(len(B)),
                          d_sd=float((A[c].mean() - B[c].mean()) / d[c].std()), p=float(stats.mannwhitneyu(A[c], B[c]).pvalue))
        return out
    N["overlap"] = {"HCPya_vs_camCAN_22_37": overlap(7, 8, 22, 37), "camCAN_vs_HCPa_36_89": overlap(8, 9, 36, 89)}
    # sweeps
    sw = sweep.merge(d[["sid", "stable_rank"]], on="sid")
    N["eta_sweep"] = {f"{e:g}": dict(MC_mean=float(g["MC_theory"].mean()), eff_rank_mean=float(g["eff_rank"].mean()),
                                     r_rs=r_(g["MC_theory"].values, g["stable_rank"].values))
                      for e, g in sw.groupby("eta_rel")}
    if ridge is not None:
        rr = ridge.merge(d[["sid", "stable_rank"]], on="sid")
        N["ridge_sweep"] = {f"{e:g}": dict(n=int(len(g)), MC_mean=float(g["MC"].mean()), r_rs=r_(g["MC"].values, g["stable_rank"].values),
                                           r_sim_theory=r_(g["MC"].values, g["MC_lin"].values))
                            for e, g in rr.groupby("ridge")}
    # node level
    nd = nodes.dropna()
    N["nodes"] = dict(n=int(len(nd)), R2_order2=float(np.corrcoef(nd["Cii"], 1 + nd["T2"])[0, 1] ** 2),
                      R2_order3=float(np.corrcoef(nd["Cii"], 1 + nd["T2"] + nd["T3"])[0, 1] ** 2))
    for lab, sel in [("k_lt20", nd["k"] < 20), ("k_20_40", nd["k"].between(20, 40)), ("k_gt40", nd["k"] > 40)]:
        x = nd[sel]
        N["nodes"][f"order3_share_{lab}"] = float((x["T3"] / (x["Cii"] - 1)).median())
        N["nodes"][f"order2_relerr_{lab}"] = float(((x["Cii"] - 1 - x["T2"]) / (x["Cii"] - 1)).median())
    N["subject_level_R2_order2"] = float(np.corrcoef(d["Cii_mean"], 1 + d["m2"] / 2)[0, 1] ** 2)
    # surrogates
    b, a = np.polyfit(d["m2"], d[MC], 1)
    sm = surr.groupby(["subject_id", "model"])[["MC", "m2", "MC_lin"]].mean().reset_index()
    wide = sm.pivot(index="subject_id", columns="model", values="MC")
    mods, pv = {}, {}
    for m in ["Real", "Uniform", "BrokenStick", "Reshuffle", "SignFlip", "RealAsym"]:
        if m not in wide: continue
        g = sm[sm["model"] == m]
        e = dict(MC=float(g["MC"].mean()), m2=float(g["m2"].mean()), MC_lin=float(g["MC_lin"].mean()),
                 MC_from_line=float(a + b * g["m2"].mean()))
        if m != "Real":
            dd = (wide[m] - wide["Real"]).dropna()
            e.update(dMC_median=float(dd.median()), n=int(len(dd)))
            pv[m] = float(stats.wilcoxon(dd).pvalue)
        mods[m] = e
    ks = list(pv); padj = stats.false_discovery_control([pv[k] for k in ks])
    for k, p, q in zip(ks, [pv[k] for k in ks], padj):
        mods[k].update(wilcoxon_p=p, p_fdr=float(q))
    N["surrogates"] = dict(line=dict(intercept=float(a), slope=float(b)), models=mods,
                           n_real=int(surr["realization"].nunique()), n_subjects=int(surr["subject_id"].nunique()))
    return N

## 4 · Run the analysis

Writes to `results/paper/data/`:

| file | content |
|---|---|
| `subject_v2.csv` | one row per connectome: chain columns + stable rank, heterogeneity factors, Gramian series, input-Gramian spectrum, closed-form MC |
| `eta_sweep.csv` | closed-form MC and effective rank for noise floors η/η₀ = 1e-4 … 1e4 |
| `examples.csv`, `examples_curves.csv` | input-Gramian eigenvalues and forgetting curves of three subjects (p5, p50, p95 of MC) |
| `nodes_v2.csv` | node-level sample (150 subjects) |
| `surrogates.csv` | copy of `results/chain/chain_surrogates_scale1e-05.csv` |
| `ridge_sweep.csv` | (`RIDGE_SWEEP`) simulated tanh ESN at four ridge penalties, 300 subjects |
| `numbers.json` | every number quoted in the manuscript and SI |

In [ ]:
def run_analysis(data, chain, out, ridge_sweep=False, ridge_n=300, threads=0, eigen=None):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    chain = Path(chain)

    print("loading connectomes ...", flush=True)
    sid, mats, meta = ca.load_pickle(data) if str(data).endswith(".pkl") else ca.load_export(data)
    cs = pd.read_csv(chain / "chain_subject.csv")
    if MC not in cs:
        raise RuntimeError(f"{chain}/chain_subject.csv has no {MC}: run the chain (§1) with scales 1 and 1e-5")

    print("per-subject quantities ...", flush=True)
    rows, sweep, ex_c, ex_m = [], [], {}, {}
    for n, (s, A) in enumerate(zip(sid, mats)):
        q, sw, c, m = subject_quantities(A, s)
        q["sid"] = int(s); rows.append(q)
        for x in sw: x["sid"] = int(s); sweep.append(x)
        ex_c[int(s)], ex_m[int(s)] = c, m
        if (n + 1) % 500 == 0: print(f"  {n + 1}/{len(sid)}", flush=True)
    new = pd.DataFrame(rows)
    keep = [c for c in cs.columns if c not in new.columns or c == "sid"]
    df = cs[keep].merge(new, on="sid", how="inner")
    sweep = pd.DataFrame(sweep)

    # three example subjects: closest to the 5th, 50th and 95th percentile of simulated MC
    ex_rows, cur_rows = [], []
    for lab, pct in [("p5", 5), ("p50", 50), ("p95", 95)]:
        target = np.percentile(df[MC].dropna(), pct)
        s = int(df.loc[(df[MC] - target).abs().idxmin(), "sid"])
        c = ex_c[s]
        ex_rows += [dict(label=lab, sid=s, k=k + 1, c_over_eta=float(v / ETA0)) for k, v in enumerate(c)]
        cur_rows += [dict(label=lab, sid=s, tau=t + 1, m=float(v)) for t, v in enumerate(ex_m[s])]

    print("node sample ...", flush=True)
    pick = np.random.default_rng(0).choice(len(sid), size=min(150, len(sid)), replace=False)
    nodes = pd.concat([node_sample(mats[i]).assign(sid=int(sid[i])) for i in pick], ignore_index=True)

    surr_src = chain / "chain_surrogates_scale1e-05.csv"
    surr = pd.read_csv(surr_src)

    ridge = None
    if ridge_sweep:
        print("ridge sweep (tanh ESN) ...", flush=True)
        work = out / "work"; work.mkdir(exist_ok=True)
        sub = np.sort(np.random.default_rng(2).choice(len(sid), min(ridge_n, len(sid)), replace=False))
        csv = work / "connectomes_ridge.csv"; ca.write_connectomes_csv(csv, sid[sub], mats[sub])
        exe = ca.build("esn_mc", ca.find_eigen(eigen))
        parts = []
        for rg in [1e-3, 1e-6, 1e-9, 1e-12]:
            o = work / f"ridge_{rg:g}.csv"
            ca.run_cfg(exe, work / f"cfg_ridge_{rg:g}.txt", {**ca.ESN_CFG, "ridge": rg, "input_scale": 1e-5,
                       "log_theory": 1, "threads": threads, "in_csv": str(csv), "out_csv": str(o)})
            t = pd.read_csv(o).rename(columns={"subject_id": "sid", "MC_Glob": "MC"})
            parts.append(t[["sid", "MC", "MC_lin"]].assign(ridge=rg))
        ridge = pd.concat(parts, ignore_index=True)
        ridge.to_csv(out / "ridge_sweep.csv", index=False)
    elif (out / "ridge_sweep.csv").exists():
        ridge = pd.read_csv(out / "ridge_sweep.csv")

    print("numbers ...", flush=True)
    N = numbers(df, sweep, nodes, surr, ridge)
    N["constants"] = dict(rho=RHO, s2u=S2U, n_train=NTR, ridge=RIDGE, eta0=ETA0, tau_max=TAU, age_split=AGE_SPLIT)

    df.to_csv(out / "subject_v2.csv", index=False)
    sweep.to_csv(out / "eta_sweep.csv", index=False)
    pd.DataFrame(ex_rows).to_csv(out / "examples.csv", index=False)
    pd.DataFrame(cur_rows).to_csv(out / "examples_curves.csv", index=False)
    nodes.to_csv(out / "nodes_v2.csv", index=False)
    shutil.copy(surr_src, out / "surrogates.csv")
    (out / "numbers.json").write_text(json.dumps(N, indent=2))
    print(f"outputs in {out}")

In [ ]:
missing = [p for p in (DATA_PKL, CHAIN_DIR / "chain_subject.csv") if not p.exists()]
if RUN_ANALYSIS and not missing:
    t0 = time.time()
    run_analysis(DATA_PKL, CHAIN_DIR, DATA, ridge_sweep=RIDGE_SWEEP, threads=THREADS, eigen=EIGEN)
    print(f"done in {time.time() - t0:.0f} s")
elif RUN_ANALYSIS:
    print("skipped, missing:", *missing, "\n(the figures below use whatever is already in", DATA, ")")

Headline numbers (all of them are in `numbers.json`):

In [ ]:
Nj = json.loads((DATA / "numbers.json").read_text())
print(f"connectomes: {Nj['n']}")
print(f"MC (input scale 1e-5): median {Nj['MC']['median']:.2f}, range {Nj['MC']['min']:.2f}-{Nj['MC']['max']:.2f}")
print(f"closed form vs simulation: r = {Nj['theory']['r_sim_cpp']:.3f} (C++ mask), {Nj['theory']['r_sim_py']:.3f} (numpy mask)")
r = Nj["stable_rank"]["r_MC"]
print(f"stable rank vs MC: r = {r['all']:.2f} (development {r['dev']:.2f}, aging {r['aging']:.2f}); "
      f"R2 log = {Nj['stable_rank']['R2_log']:.2f}")
print(f"effective rank vs MC: r = {Nj['effective_rank']['r_MC']:.3f}")
print("surrogates, median dMC (null - real):",
      {m: round(v["dMC_median"], 2) for m, v in Nj["surrogates"]["models"].items() if "dMC_median" in v})

## 5 · Figures

Drawn from `results/paper/data/` only, one function per figure, saved to
`results/paper/figures/*.pdf, *.png` (PNAS widths: 7.0 in double column, 3.42 in single column).

In [ ]:
MCL = "MClin_s1e-05"
N_NODES = 90

# cohort colours of the manuscript figures (matplotlib tab10 order, purple skipped)
COHORTS = {1: ("dHCP", "#1f77b4"), 2: ("BCP", "#ff7f0e"), 3: ("CALM", "#2ca02c"), 4: ("RED", "#d62728"),
           5: ("ACE", "#8c564b"), 6: ("HCPd", "#e377c2"), 7: ("HCPya", "#7f7f7f"), 8: ("camCAN", "#bcbd22"),
           9: ("HCPa", "#17becf")}
# surrogate colours (Okabe-Ito), real network in black
MODELS = {"Real": ("Real", "#000000"), "Uniform": ("Uniform", "#E69F00"), "BrokenStick": ("Broken stick", "#56B4E9"),
          "Reshuffle": ("Reshuffle", "#009E73"), "SignFlip": ("Sign flip", "#CC79A7"),
          "RealAsym": ("Directed control", "#D55E00")}
EXAMPLES = {"p5": "#9ecae1", "p50": "#4292c6", "p95": "#08306b"}          # one hue, light -> dark
INK, MUTED, GRID = "#222222", "#6b6b6b", "#e6e6e6"
W2, W1 = 7.0, 3.42                                                        # PNAS widths (inches)

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"], "font.size": 7,
    "axes.labelsize": 7, "axes.titlesize": 7, "xtick.labelsize": 6, "ytick.labelsize": 6,
    "legend.fontsize": 6, "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 1.2, "savefig.dpi": 300, "pdf.fonttype": 42, "legend.frameon": False,
    "mathtext.fontset": "cm",
})

Data and plotting helpers. `save()` writes the PDF and PNG and shows the figure here.

In [ ]:
def load():
    D = dict(s=pd.read_csv(DATA / "subject_v2.csv"), eta=pd.read_csv(DATA / "eta_sweep.csv"),
             ex=pd.read_csv(DATA / "examples.csv"), cur=pd.read_csv(DATA / "examples_curves.csv"),
             nodes=pd.read_csv(DATA / "nodes_v2.csv"), surr=pd.read_csv(DATA / "surrogates.csv"),
             N=json.loads((DATA / "numbers.json").read_text()))
    D["ridge"] = pd.read_csv(DATA / "ridge_sweep.csv") if (DATA / "ridge_sweep.csv").exists() else None
    D["s"] = D["s"].dropna(subset=[MC])
    return D

In [ ]:
def save(fig, name):
    OUT.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(OUT / f"{name}.png", bbox_inches="tight", dpi=200)
    display(fig)
    plt.close(fig)

In [ ]:
def label(ax, letter, x=-0.2, y=1.06):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=8, fontweight="bold", va="bottom", ha="left", color=INK)


def note(ax, text, x=0.04, y=0.96, ha="left", va="top"):
    ax.text(x, y, text, transform=ax.transAxes, fontsize=6, ha=ha, va=va, color=INK)


def grid(ax, axis="both"):
    ax.grid(True, axis=axis, color=GRID, lw=0.5, zorder=0)
    ax.set_axisbelow(True)


def cohort_legend(ax, loc="lower right", ncol=1, codes=None):
    codes = codes or list(COHORTS)
    h = [Line2D([], [], marker="o", ls="", ms=3.5, mfc=COHORTS[c][1], mec="white", mew=0.3) for c in codes]
    ax.legend(h, [COHORTS[c][0] for c in codes], loc=loc, ncol=ncol, handletextpad=0.1, columnspacing=0.6,
              borderaxespad=0.2, labelspacing=0.25)


def scatter_cohorts(ax, d, x, y, s=2.5, alpha=0.55):
    for c, (nm, col) in COHORTS.items():
        g = d[d["dataset"] == c]
        ax.scatter(g[x], g[y], s=s, color=col, alpha=alpha, lw=0, rasterized=True)


def age_means(ax, d, col, ms=2.6):
    """Mean of `col` per cohort and age (1-y bins below 40 y, 3-y bins above), as in the v1 figures."""
    for c, (nm, colr) in COHORTS.items():
        g = d[d["dataset"] == c]
        if g.empty: continue
        a = g["age"].values
        b = np.where(a < 40, np.floor(a), 40 + 3 * np.floor((a - 40) / 3) + 1)
        m = g.groupby(b)[col].mean()
        ax.plot(m.index, m.values, "-o", color=colr, ms=ms, lw=0.7, mec="white", mew=0.25)
    ax.axvline(AGE_SPLIT, color=MUTED, lw=0.6, ls=":", zorder=0)


def rank_axis(ax):
    """Log x-axis for the stable rank with plain tick labels."""
    ax.set_xscale("log")
    ax.set_xticks([3, 4, 5, 6, 8, 10]); ax.set_xticklabels(["3", "4", "5", "6", "8", "10"])
    ax.xaxis.set_minor_formatter(NullFormatter())


def corr(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    return np.corrcoef(x[m], y[m])[0, 1]


def binned(x, y, edges):
    """Median and interquartile range of y in bins of x."""
    idx = np.digitize(x, edges) - 1
    rows = []
    for b in range(len(edges) - 1):
        v = y[idx == b]
        if len(v) >= 20:
            rows.append((0.5 * (edges[b] + edges[b + 1]), *np.percentile(v, [25, 50, 75])))
    return np.array(rows)


def null_upsilon(k):
    mu = 2 * k / (k + 1)
    var = k ** 2 * ((20 + 4 * k) / ((k + 1) * (k + 2) * (k + 3)) - 4 / (k + 1) ** 2)
    return mu, np.sqrt(np.clip(var, 0, None))


def surrogate_means(D):
    sm = D["surr"].groupby(["subject_id", "model"])[["MC", "m2", "MC_lin"]].mean().reset_index()
    return sm.groupby("model").agg(MC=("MC", "mean"), MCsd=("MC", "std"), m2=("m2", "mean"), m2sd=("m2", "std"),
                                   MC_lin=("MC_lin", "mean"))

In [ ]:
D = load()
print(f"{len(D['s'])} connectomes with MC")

### Figure 2 · From local disparity to the spectrum
(a) disparity against degree with the broken-stick null; (b) accuracy of the order-2 and order-3 truncations of the subgraph centrality; (c) subgraph centrality and stable rank; (d) local and global heterogeneity.

In [ ]:
def fig2_order2(D):
    """From local disparity to the spectrum."""
    s, nd = D["s"], D["nodes"].dropna()
    fig, axs = plt.subplots(1, 4, figsize=(W2, 1.85), gridspec_kw=dict(wspace=0.55))

    ax = axs[0]                                         # (a) disparity against degree
    ax.scatter(nd["k"], nd["Ups"], s=1.2, color="#bdbdbd", lw=0, alpha=0.5, rasterized=True)
    k = np.arange(2, int(nd["k"].max()) + 1)
    mu, sd = null_upsilon(k.astype(float))
    ax.fill_between(k, mu - 2 * sd, mu + 2 * sd, color="#9e9e9e", alpha=0.35, lw=0, label="null $\\pm2\\sigma$")
    ax.plot(k, mu, color=MUTED, lw=0.8)
    ax.plot(k, k, "--", color="#d62728", lw=0.8, label="$\\Upsilon=k$")
    e = np.unique(np.round(np.logspace(np.log10(3), np.log10(nd["k"].max()), 14)))
    bm = binned(nd["k"].values, nd["Ups"].values, e)
    ax.plot(bm[:, 0], bm[:, 2], color="#1f4e79", lw=1.4, label="median")
    ax.set(xscale="log", yscale="log", xlabel="degree $k$", ylabel="disparity $\\Upsilon=kY$")
    ax.legend(loc="upper left", handlelength=1.4)
    label(ax, "a", x=-0.32)

    ax = axs[1]                                         # (b) accuracy of the truncations by degree
    rel2 = (nd["Cii"] - 1 - nd["T2"]) / (nd["Cii"] - 1)
    rel3 = (nd["Cii"] - 1 - nd["T2"] - nd["T3"]) / (nd["Cii"] - 1)
    edges = np.arange(0, nd["k"].max() + 8, 8)
    for rel, nm, col in [(rel2, "order 2", "#1f4e79"), (rel3, "order 3", "#6baed6")]:
        bm = binned(nd["k"].values, 100 * rel.values, edges)
        ax.fill_between(bm[:, 0], bm[:, 1], bm[:, 3], color=col, alpha=0.25, lw=0)
        ax.plot(bm[:, 0], bm[:, 2], "-o", color=col, ms=2.5, lw=1.0, label=nm)
    ax.set(xlabel="degree $k$", ylabel="error in $C_{ii}-1$ (%)", ylim=(0, None))
    grid(ax, "y"); ax.legend(loc="upper left")
    label(ax, "b", x=-0.3)

    ax = axs[2]                                         # (c) subgraph centrality and stable rank
    ax.scatter(s["stable_rank"], s["Cii_mean"] - 1, s=2, color="#4d4d4d", alpha=0.4, lw=0, rasterized=True)
    x = np.linspace(s["stable_rank"].min(), s["stable_rank"].max(), 10)
    ax.plot(x, 0.9801 * x / (2 * N_NODES), color="#d62728", lw=0.8, ls="--", label="order-2 term")
    ax.set(xlabel="stable rank $r_s$", ylabel="$\\langle C_{ii}\\rangle-1$")
    note(ax, f"$r={corr(s['stable_rank'].values, s['Cii_mean'].values):.3f}$")
    ax.legend(loc="lower right")
    label(ax, "c", x=-0.32)

    ax = axs[3]                                         # (d) local and global heterogeneity
    sc = ax.scatter(s["Ratio"], s["s2t"], c=s["stable_rank"], cmap="Blues", s=2, lw=0, rasterized=True,
                    vmin=s["stable_rank"].quantile(0.01) - 1.5, vmax=s["stable_rank"].quantile(0.99))
    ax.set(xlabel="local: $\\langle\\Upsilon/\\Upsilon_{null}\\rangle$", ylabel="global: $\\langle\\tilde s^{2}\\rangle$")
    cb = fig.colorbar(sc, ax=ax, fraction=0.05, pad=0.02); cb.set_label("$r_s$", fontsize=6)
    cb.ax.tick_params(labelsize=5); cb.outline.set_linewidth(0.4)
    label(ax, "d", x=-0.32)
    save(fig, "fig2_order2")


fig2_order2(D)

### Figure 4 · Memory counts input directions above the noise floor
(a) closed form against simulation; (b) input-Gramian spectra and the noise floor; (c) the stable rank predicts memory; (d) energy versus dimension; (e) surrogates on the stable-rank line.

In [ ]:
def fig4_memory(D):
    """Memory counts input directions above the noise floor; the stable rank sets the count."""
    s, N = D["s"], D["N"]
    fig = plt.figure(figsize=(W2, 3.9))
    gs = fig.add_gridspec(2, 6, hspace=0.62, wspace=1.6)
    axa, axb, axc = fig.add_subplot(gs[0, 0:2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[0, 4:6])
    axd, axe = fig.add_subplot(gs[1, 0:3]), fig.add_subplot(gs[1, 3:6])

    ax = axa                                            # (a) closed form against simulation
    ax.scatter(s[MCL], s[MC], s=2, color="#4d4d4d", alpha=0.4, lw=0, rasterized=True)
    lo, hi = s[[MC, MCL]].min().min() - 0.2, s[[MC, MCL]].max().max() + 0.2
    ax.plot([lo, hi], [lo, hi], color="#d62728", lw=0.8, ls="--")
    ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel="MC, closed form", ylabel="MC, simulated reservoir")
    note(ax, f"$r={N['theory']['r_sim_cpp']:.3f}$\n3,901 connectomes")
    label(ax, "a")

    ax = axb                                            # (b) input-Gramian spectra and the noise floor
    ex = D["ex"]
    for lab, g in ex.groupby("label"):
        g = g[g["k"] <= 22]
        mc = s.loc[s["sid"] == g["sid"].iloc[0], MC].iloc[0]
        nab = int((g["c_over_eta"] > 1).sum())
        ax.plot(g["k"], g["c_over_eta"], "-o", color=EXAMPLES[lab], ms=2.2, lw=0.9,
                label=f"MC = {mc:.1f}: {nab} directions")
    ax.axhline(1, color=INK, lw=0.8, ls="--")
    ax.text(22, 3, "noise floor $\\eta$", fontsize=6, ha="right", va="bottom", color=INK)
    ax.set(yscale="log", xlim=(0, 22.5), ylim=(1e-3, 1e12), xlabel="direction $k$ of the input Gramian",
           ylabel="variance $c_k/\\eta$")
    ax.set_yticks([1e-3, 1, 1e3, 1e6, 1e9, 1e12])
    ax.legend(loc="upper right", handlelength=1.2, fontsize=5.5)
    label(ax, "b")

    ax = axc                                            # (c) stable rank predicts memory
    scatter_cohorts(ax, s, "stable_rank", MC, s=2)
    x = np.linspace(s["stable_rank"].min(), s["stable_rank"].max(), 50)
    p = np.polyfit(np.log(s["stable_rank"]), s[MC], 1)
    ax.plot(x, np.polyval(p, np.log(x)), color=INK, lw=0.9)
    rank_axis(ax); ax.set(xlabel="stable rank $r_s$", ylabel="MC, simulated")
    r = N["stable_rank"]["r_MC"]
    note(ax, f"$r={r['all']:.2f}$ (dev. {r['dev']:.2f}, aging {r['aging']:.2f})", y=0.98)
    cohort_legend(ax, loc="lower right", ncol=2)
    ax.get_legend().set_bbox_to_anchor((1.04, -0.03))
    for t in ax.get_legend().get_texts(): t.set_fontsize(5)
    label(ax, "c")

    ax = axd                                            # (d) energy versus dimension
    T = ["1", "2", "3", "5", "10", "20", "$\\infty$"]
    rv = [N["gram_series_r_MC"][f"T{t}"] for t in [1, 2, 3, 5, 10, 20]] + [N["gram_series_r_MC"]["Tinf"]]
    xs = np.arange(len(T))
    ax.bar(xs, rv, width=0.6, color=["#1f4e79"] + ["#6baed6"] * 5 + ["#bdbdbd"], edgecolor="white", lw=0.5)
    for xi, v in zip(xs, rv):
        ax.text(xi, v + (0.03 if v >= 0 else -0.03), f"{v:.2f}", ha="center", va="bottom" if v >= 0 else "top",
                fontsize=5.5, color=INK)
    ax.axhline(0, color=MUTED, lw=0.6)
    T[0], T[-1] = "1\n(stable rank)", "$\\infty$\n(avg. controllability)"
    ax.set_xticks(xs); ax.set_xticklabels(T)
    ax.set(ylim=(-0.3, 1.05), xlabel="terms $T$ of the Gramian series $S_T$", ylabel="$r$ with MC")
    grid(ax, "y")
    label(ax, "d", x=-0.12)

    ax = axe                                            # (e) surrogates on the stable-rank line
    ax.scatter(s["m2"], s[MC], s=1.5, color="#d9d9d9", lw=0, rasterized=True)
    x = np.linspace(0.025, 0.09, 10); L = N["surrogates"]["line"]
    ax.plot(x, L["intercept"] + L["slope"] * x, color=MUTED, lw=0.8)
    sm = surrogate_means(D)
    for m, (nm, col) in MODELS.items():
        if m not in sm.index: continue
        r_ = sm.loc[m]
        ax.plot(r_["m2"], r_["MC"], "o", ms=5, color=col, mec="white", mew=0.6, zorder=3)
        dx, dy, ha = {"Real": (-0.002, 0.45, "right"), "Uniform": (0.002, -0.4, "left"),
                      "BrokenStick": (0.002, -0.45, "left"), "Reshuffle": (0.0015, 0.55, "left"),
                      "SignFlip": (0.003, -0.45, "left"), "RealAsym": (0.003, 0.0, "left")}[m]
        ax.text(r_["m2"] + dx, r_["MC"] + dy, nm, fontsize=5.5, ha=ha, va="center", color=INK)
    ax.set(xlabel="$m_2=\\mathrm{Tr}\\,\\tilde W^2/N$ (stable rank / $N$)", ylabel="MC, simulated",
           xlim=(0.02, 0.095))
    label(ax, "e", x=-0.12)
    save(fig, "fig4_memory")


fig4_memory(D)

### Figure 5 · Lifespan
Memory, stable rank and the two heterogeneity factors across the lifespan; (c) camCAN alone (single site).

In [ ]:
def fig5_lifespan(D):
    """Lifespan of memory, stable rank and the two heterogeneity factors."""
    s, N = D["s"], D["N"]
    fig, axs = plt.subplots(2, 3, figsize=(W2, 3.9), gridspec_kw=dict(hspace=0.55, wspace=0.45, top=0.9))
    panels = [(axs[0, 0], MC, "MC", "a"), (axs[0, 1], "stable_rank", "stable rank $r_s$", "b"),
              (axs[1, 0], "Ratio", "local: $\\langle\\Upsilon/\\Upsilon_{null}\\rangle$", "d"),
              (axs[1, 1], "s2t", "global: $\\langle\\tilde s^{2}\\rangle$", "e"),
              (axs[1, 2], "strength", "raw strength (streamlines)", "f")]
    for ax, col, yl, let in panels:
        ax.scatter(s["age"], s[col], s=1, color="#e0e0e0", lw=0, rasterized=True, zorder=1)
        age_means(ax, s, col)
        ax.set(xlabel="age (y)", ylabel=yl, xlim=(-3, 93))
        label(ax, let)
    ages = N["age_correlations"]
    for ax, col in [(axs[0, 0], MC), (axs[0, 1], "stable_rank"), (axs[1, 0], "Ratio"), (axs[1, 1], "s2t"),
                    (axs[1, 2], "strength")]:
        note(ax, f"$r_{{age}}$: {ages['dev'][col]:+.2f} | {ages['aging'][col]:+.2f}", x=0.98, y=0.98, ha="right")
    h = [Line2D([], [], marker="o", ls="-", lw=0.7, ms=3, mfc=c[1], color=c[1], mec="white", mew=0.3) for c in COHORTS.values()]
    fig.legend(h, [c[0] for c in COHORTS.values()], loc="upper center", ncol=9, bbox_to_anchor=(0.5, 1.0),
               handletextpad=0.3, columnspacing=1.0)

    ax = axs[0, 2]                                      # (c) camCAN alone
    cc = s[s["dataset"] == 8]
    bins = [18, 25, 32, 40, 50, 60, 70, 80, 90]
    g = cc.groupby(pd.cut(cc["age"], bins), observed=True)
    xm, ym, ye = g["age"].mean(), g[MC].mean(), g[MC].sem()
    ax.scatter(cc["age"], cc[MC], s=1.5, color="#e5e5a8", lw=0, rasterized=True)
    q = N["camcan"]["MC_quad"]
    xa = np.linspace(19, 89, 100)
    ax.axvspan(*q["ci"], color="#bcbd22", alpha=0.15, lw=0)
    ax.plot(xa, np.polyval(q["coef"], xa), color=INK, lw=0.9)
    ax.errorbar(xm, ym, yerr=ye, fmt="o", ms=3, color="#8c8d10", mec="white", mew=0.4, elinewidth=0.7)
    ax.axvline(q["vertex"], color=INK, lw=0.6, ls=":")
    ax.set(xlabel="age (y)", ylabel="MC (camCAN only)", xlim=(15, 92), ylim=(cc[MC].quantile(0.01), cc[MC].quantile(0.995)))
    ax.text(0.04, 0.04, f"single site, n = {len(cc)}\nminimum {q['vertex']:.1f} y (95% CI {q['ci'][0]:.0f}-{q['ci'][1]:.0f})",
            transform=ax.transAxes, fontsize=6, va="bottom", color=INK,
            bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none", alpha=0.85))
    label(ax, "c")
    save(fig, "fig5_lifespan")


fig5_lifespan(D)

### SI · Closed-form theory against simulation
Linear and nonlinear regime, and closed-form forgetting curves of three subjects.

In [ ]:
def figS_theory(D):
    s, cur, N = D["s"], D["cur"], D["N"]
    fig, axs = plt.subplots(1, 3, figsize=(W2, 2.0), gridspec_kw=dict(wspace=0.45))
    for ax, (x, y, ttl, let) in zip(axs[:2], [(MCL, MC, "$\\sigma_{in}=10^{-5}$ (linear)", "a"),
                                              ("MClin_s1", "MC_s1", "$\\sigma_{in}=1$ (nonlinear)", "b")]):
        ax.scatter(s[x], s[y], s=2, color="#4d4d4d", alpha=0.4, lw=0, rasterized=True)
        lo, hi = min(s[x].min(), s[y].min()) - 0.2, max(s[x].max(), s[y].max()) + 0.2
        ax.plot([lo, hi], [lo, hi], color="#d62728", lw=0.8, ls="--")
        ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel="MC, closed form", ylabel="MC, simulated", title=ttl)
        note(ax, f"$r={corr(s[x].values, s[y].values):.3f}$")
        label(ax, let)
    ax = axs[2]
    for lab, g in cur.groupby("label"):
        mc = s.loc[s["sid"] == g["sid"].iloc[0], MC].iloc[0]
        ax.plot(g["tau"], g["m"], "-o", color=EXAMPLES[lab], ms=2.2, lw=0.9, label=f"MC = {mc:.1f}")
    ax.set(xlabel="delay $\\tau$", ylabel="$m(\\tau)$, closed form", ylim=(0, 1.05), xticks=[1, 5, 10, 15, 20])
    grid(ax, "y"); ax.legend(loc="upper right")
    label(ax, "c")
    save(fig, "figS_theory")


figS_theory(D)

### SI · Input-Gramian spectrum
Eigenvalues against the noise floor, effective rank, decay ratio.

In [ ]:
def figS_gramian(D):
    s, ex, N = D["s"], D["ex"], D["N"]
    fig, axs = plt.subplots(1, 3, figsize=(W2, 2.0), gridspec_kw=dict(wspace=0.45))
    ax = axs[0]
    for lab, g in ex.groupby("label"):
        g = g[g["k"] <= 24]
        mc = s.loc[s["sid"] == g["sid"].iloc[0], MC].iloc[0]
        ax.plot(g["k"], g["c_over_eta"], "-o", color=EXAMPLES[lab], ms=2, lw=0.9, label=f"MC = {mc:.1f}")
    ax.axhline(1, color=INK, lw=0.8, ls="--")
    ax.axhspan(1e-8, 1e-4, color="#f0f0f0", lw=0, zorder=0)
    ax.text(23.5, 2e-6, "double-precision limit", fontsize=5.5, ha="right", va="center", color=MUTED)
    ax.set(yscale="log", xlabel="direction $k$", ylabel="$c_k/\\eta_0$", xlim=(0, 24), ylim=(1e-8, 1e12))
    ax.legend(loc="upper right"); label(ax, "a")
    ax = axs[1]
    scatter_cohorts(ax, s, "eff_rank", MC, s=2)
    ax.set(xlabel="effective rank $\\sum_k c_k/(c_k+\\eta_0)$", ylabel="MC, simulated")
    note(ax, f"$r={N['effective_rank']['r_MC']:.3f}$"); label(ax, "b")
    ax = axs[2]
    scatter_cohorts(ax, s, "stable_rank", "q_decay", s=2)
    rank_axis(ax); ax.set(xlabel="stable rank $r_s$", ylabel="decay ratio $q$")
    note(ax, f"$r={N['effective_rank']['r_q_rs']:.2f}$", x=0.96, y=0.04, ha="right", va="bottom")
    cohort_legend(ax, loc="upper left", ncol=2)
    label(ax, "c")
    save(fig, "figS_gramian")


figS_gramian(D)

### SI · Noise floor
Closed-form MC and its correlation with the stable rank across noise floors, with the simulated ridge sweep when available.

In [ ]:
def figS_noise(D):
    eta, ridge, N = D["eta"], D["ridge"], D["N"]
    s = D["s"][["sid", "stable_rank"]]
    fig, axs = plt.subplots(1, 2, figsize=(W1 * 1.6, 2.0), gridspec_kw=dict(wspace=0.45))
    g = eta.groupby("eta_rel")["MC_theory"]
    xs = np.array(sorted(eta["eta_rel"].unique()))
    q = g.quantile([0.25, 0.5, 0.75]).unstack().loc[xs]
    ax = axs[0]
    ax.fill_between(xs, q[0.25], q[0.75], color="#6baed6", alpha=0.3, lw=0)
    ax.plot(xs, g.mean().loc[xs], "-", color="#1f4e79", lw=1.2, label="closed form, 3,901")
    rr = [N["eta_sweep"][f"{x:g}"]["r_rs"] for x in xs]
    axs[1].plot(xs, rr, "-", color="#1f4e79", lw=1.2, label="closed form")
    if ridge is not None:
        # ridge penalty (relative to sigma_in^2) -> noise floor relative to the main text (ridge 1e-6)
        rs = ridge.merge(s, on="sid")
        for rg, h in rs.groupby("ridge"):
            x = rg / 1e-6
            bad = rg < 1e-10
            col = "#bdbdbd" if bad else "#d62728"
            ax.errorbar(x, h["MC"].mean(), yerr=h["MC"].std(), fmt="o", ms=3.5, color=col, mec="white", mew=0.4,
                        elinewidth=0.7)
            axs[1].plot(x, corr(h["MC"].values, h["stable_rank"].values), "o", ms=3.5, color=col, mec="white", mew=0.4)
        for a in axs:
            a.axvspan(1e-7, 1e-5, color="#f0f0f0", lw=0, zorder=0)
        h1 = [Line2D([], [], color="#1f4e79", lw=1.2), Line2D([], [], marker="o", ls="", color="#d62728", ms=3.5),
              Line2D([], [], marker="o", ls="", color="#bdbdbd", ms=3.5)]
        axs[1].legend(h1, ["closed form", "simulated (300)", "rounding-limited"], loc="lower left", numpoints=1)
    ax.axvline(1, color=MUTED, lw=0.6, ls=":")
    axs[1].axvline(1, color=MUTED, lw=0.6, ls=":")
    ax.set(xscale="log", xlabel="noise floor $\\eta/\\eta_0$", ylabel="mean MC")
    axs[1].set(xscale="log", xlabel="noise floor $\\eta/\\eta_0$", ylabel="$r$(MC, stable rank)", ylim=(0.5, 1.0))
    for a in axs: grid(a)
    label(axs[0], "a"); label(axs[1], "b")
    save(fig, "figS_noise")


figS_noise(D)

### SI · Energy versus dimension
Truncated Gramian series and average controllability.

In [ ]:
def figS_energy(D):
    s, N = D["s"], D["N"]
    fig, axs = plt.subplots(1, 2, figsize=(W1 * 1.7, 2.1), gridspec_kw=dict(wspace=0.55))
    ax = axs[0]
    T = [1, 2, 3, 5, 10, 20]
    rv = [N["gram_series_r_MC"][f"T{t}"] for t in T]
    ax.plot(T, rv, "-o", color="#1f4e79", ms=3, lw=1)
    ax.axhline(N["gram_series_r_MC"]["Tinf"], color=MUTED, lw=0.8, ls="--")
    ax.text(20, N["gram_series_r_MC"]["Tinf"] + 0.03, "$T=\\infty$", fontsize=6, ha="right", va="bottom", color=MUTED)
    ax.axhline(0, color=MUTED, lw=0.5)
    ax.set(xscale="log", xlabel="terms $T$", ylabel="$r(S_T,\\mathrm{MC})$", xticks=T)
    ax.set_xticklabels([str(t) for t in T]); grid(ax, "y")
    label(ax, "a")
    ax = axs[1]
    sc = ax.scatter(s["stable_rank"], s[MC], c=s["gram_Tinf"], cmap="Blues", s=2, lw=0, rasterized=True,
                    vmin=s["gram_Tinf"].quantile(0.01) * 0.6, vmax=s["gram_Tinf"].quantile(0.99))
    rank_axis(ax); ax.set(xlabel="stable rank $r_s$", ylabel="MC, simulated")
    cb = fig.colorbar(sc, ax=ax, fraction=0.05, pad=0.02); cb.set_label("$S_\\infty$ (energy)", fontsize=6)
    cb.ax.tick_params(labelsize=5); cb.outline.set_linewidth(0.4)
    note(ax, f"partial $r(S_\\infty,\\mathrm{{MC}}\\,|\\,r_s)={N['predictors']['all']['avg_controllability']['partial_given_rs']:.2f}$")
    label(ax, "b")
    save(fig, "figS_energy")


figS_energy(D)

### SI · The global factor

In [ ]:
def figS_global(D):
    s, N = D["s"], D["N"]
    fig, axs = plt.subplots(1, 4, figsize=(W2, 1.9), gridspec_kw=dict(wspace=0.5))
    for ax, (x, xl, let) in zip(axs[:2], [("cv_s", "CV of node strength", "a"),
                                          ("ipr_v1", "IPR of leading eigenvector", "b")]):
        scatter_cohorts(ax, s, x, "s2t", s=2)
        ax.set(xlabel=xl, ylabel="$\\langle\\tilde s^{2}\\rangle$")
        note(ax, f"$r={corr(s[x].values, s['s2t'].values):.2f}$", x=0.96, ha="right")
        label(ax, let, x=-0.3)
    axs[1].set_xscale("log"); axs[1].xaxis.set_minor_formatter(NullFormatter())
    axs[1].set_xticks([0.02, 0.05, 0.1]); axs[1].set_xticklabels(["0.02", "0.05", "0.1"])
    for ax, (col, yl, let) in zip(axs[2:], [("strength", "raw strength", "c"), ("s2t", "$\\langle\\tilde s^{2}\\rangle$", "d")]):
        ax.scatter(s["age"], s[col], s=1, color="#e0e0e0", lw=0, rasterized=True)
        age_means(ax, s, col, ms=2)
        ax.set(xlabel="age (y)", ylabel=yl)
        label(ax, let, x=-0.3)
    h = [Line2D([], [], marker="o", ls="", ms=3, mfc=c[1], mec="white", mew=0.3) for c in COHORTS.values()]
    fig.legend(h, [c[0] for c in COHORTS.values()], loc="lower center", ncol=9, bbox_to_anchor=(0.5, 0.98),
               handletextpad=0.1, columnspacing=0.9)
    save(fig, "figS_global")


figS_global(D)

### SI · Surrogates

In [ ]:
def figS_surrogates(D):
    surr, N = D["surr"], D["N"]
    sm = surr.groupby(["subject_id", "model"])["MC"].mean().unstack()
    fig, axs = plt.subplots(1, 2, figsize=(W2 * 0.8, 2.2), gridspec_kw=dict(wspace=0.4, width_ratios=[1.1, 1]))
    ax = axs[0]
    order = [m for m in ["Uniform", "BrokenStick", "Reshuffle", "SignFlip", "RealAsym"] if m in sm]
    data = [(sm[m] - sm["Real"]).dropna().values for m in order]
    bp = ax.boxplot(data, widths=0.55, patch_artist=True, showfliers=False, medianprops=dict(color=INK, lw=0.9),
                    whiskerprops=dict(lw=0.6, color=MUTED), capprops=dict(lw=0.6, color=MUTED))
    for b, m in zip(bp["boxes"], order):
        b.set(facecolor=MODELS[m][1], alpha=0.8, lw=0.5, edgecolor="white")
    rng = np.random.default_rng(0)
    for i, v in enumerate(data):
        ax.scatter(i + 1 + rng.uniform(-0.15, 0.15, len(v)), v, s=1.5, color=INK, alpha=0.35, lw=0, zorder=3)
    ax.axhline(0, color=MUTED, lw=0.6, ls="--")
    ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels([MODELS[m][0] for m in order], rotation=30, ha="right")
    ax.set(ylabel="$\\Delta$MC (surrogate $-$ real)")
    grid(ax, "y"); label(ax, "a")
    ax = axs[1]
    s = D["s"]
    ax.scatter(s["m2"], s[MC], s=1.5, color="#d9d9d9", lw=0, rasterized=True)
    x = np.linspace(0.025, 0.09, 10); L = N["surrogates"]["line"]
    ax.plot(x, L["intercept"] + L["slope"] * x, color=MUTED, lw=0.8)
    means = surrogate_means(D)
    for m, (nm, col) in MODELS.items():
        if m not in means.index: continue
        r_ = means.loc[m]
        ax.plot(r_["m2"], r_["MC"], "o", ms=4.5, color=col, mec="white", mew=0.5, zorder=3)
        ax.plot(r_["m2"], r_["MC_lin"], "o", ms=7, mfc="none", mec=col, mew=0.7, zorder=3)
    h = [Line2D([], [], marker="o", ls="", ms=4, color=MODELS[m][1]) for m in MODELS] + \
        [Line2D([], [], marker="o", ls="", ms=6, mfc="none", mec=MUTED)]
    ax.legend(h, [MODELS[m][0] for m in MODELS] + ["closed form"], loc="lower right", fontsize=5.5)
    ax.set(xlabel="$m_2$", ylabel="MC", xlim=(0.02, 0.095))
    label(ax, "b")
    save(fig, "figS_surrogates")


figS_surrogates(D)

### SI · Walk expansion of the subgraph centrality

In [ ]:
def figS_taylor(D):
    nd = D["nodes"].dropna()
    fig, axs = plt.subplots(1, 2, figsize=(W1 * 1.6, 2.0), gridspec_kw=dict(wspace=0.45))
    ax = axs[0]
    edges = np.arange(0, nd["k"].max() + 8, 8)
    for rel, nm, col in [((nd["Cii"] - 1 - nd["T2"]) / (nd["Cii"] - 1), "order 2", "#1f4e79"),
                         ((nd["Cii"] - 1 - nd["T2"] - nd["T3"]) / (nd["Cii"] - 1), "order 3", "#6baed6")]:
        bm = binned(nd["k"].values, 100 * rel.values, edges)
        ax.fill_between(bm[:, 0], bm[:, 1], bm[:, 3], color=col, alpha=0.25, lw=0)
        ax.plot(bm[:, 0], bm[:, 2], "-o", color=col, ms=2.5, lw=1.0, label=nm)
    ax.set(xlabel="degree $k$", ylabel="relative error in $C_{ii}-1$ (%)", ylim=(0, None))
    grid(ax, "y"); ax.legend(loc="upper left"); label(ax, "a")
    ax = axs[1]
    ax.scatter(1 + nd["T2"], nd["Cii"], s=1, color="#4d4d4d", alpha=0.3, lw=0, rasterized=True)
    lo, hi = nd["Cii"].min(), nd["Cii"].max()
    ax.plot([lo, hi], [lo, hi], color="#d62728", lw=0.8, ls="--")
    ax.set(xlabel="$1+\\frac{1}{2}\\tilde s_i^2Y_i$", ylabel="$C_{ii}$")
    note(ax, f"$R^2={D['N']['nodes']['R2_order2']:.4f}$\n{len(nd):,} nodes")
    label(ax, "b")
    save(fig, "figS_taylor")


figS_taylor(D)

### SI · Cohorts
Stable rank against MC per cohort, MC against age within cohort, camCAN minima.

In [ ]:
def figS_cohorts(D):
    s, N = D["s"], D["N"]
    fig = plt.figure(figsize=(W2, 4.3))
    gs = fig.add_gridspec(3, 6, hspace=0.75, wspace=1.4)
    xl = (s["stable_rank"].min() * 0.95, s["stable_rank"].max() * 1.05)
    yl = (s[MC].min() - 0.2, s[MC].max() + 0.2)
    for i, (c, (nm, col)) in enumerate(COHORTS.items()):
        ax = fig.add_subplot(gs[i // 3, i % 3])
        g = s[s["dataset"] == c]
        ax.scatter(g["stable_rank"], g[MC], s=2, color=col, alpha=0.6, lw=0, rasterized=True)
        rank_axis(ax); ax.set(xlim=xl, ylim=yl)
        ax.set_xticks([3, 5, 10]); ax.set_xticklabels(["3", "5", "10"] if i // 3 == 2 else [])
        if i % 3: ax.set_yticklabels([])
        ax.set_title(f"{nm}  $R^2$={N['cohorts'][nm]['R2_log_rs']:.2f}", fontsize=6, pad=2)
        if i == 7: ax.set_xlabel("stable rank $r_s$")
        if i == 3: ax.set_ylabel("MC")
        if i == 0: label(ax, "a", x=-0.45, y=1.12)
    ax = fig.add_subplot(gs[0, 3:6])
    names = [COHORTS[c][0] for c in COHORTS if "r_MC_age" in N["cohorts"][COHORTS[c][0]]]
    vals = [N["cohorts"][n]["r_MC_age"] for n in names]
    cols = [COHORTS[c][1] for c in COHORTS if COHORTS[c][0] in names]
    ax.bar(range(len(names)), vals, color=cols, width=0.65, edgecolor="white", lw=0.5)
    ax.axhline(0, color=MUTED, lw=0.6)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=35, ha="right")
    ax.set(ylabel="$r$(MC, age) within cohort"); grid(ax, "y"); label(ax, "b", x=-0.12)
    cc = s[s["dataset"] == 8]
    bins = [18, 25, 32, 40, 50, 60, 70, 80, 90]
    g = cc.groupby(pd.cut(cc["age"], bins), observed=True)
    for j, (col, yl_, key, let) in enumerate([(MC, "MC (camCAN)", "MC_quad", "c"), ("stable_rank", "$r_s$ (camCAN)", "rs_quad", "d")]):
        ax = fig.add_subplot(gs[1 + j, 3:6])
        q = N["camcan"][key]; xa = np.linspace(19, 89, 100)
        ax.axvspan(*q["ci"], color="#bcbd22", alpha=0.15, lw=0)
        ax.plot(xa, np.polyval(q["coef"], xa), color=INK, lw=0.9)
        ax.errorbar(g["age"].mean(), g[col].mean(), yerr=g[col].sem(), fmt="o", ms=3, color="#8c8d10", mec="white",
                    mew=0.4, elinewidth=0.7)
        ax.axvline(q["vertex"], color=INK, lw=0.6, ls=":")
        ax.set(ylabel=yl_, xlim=(15, 92)); grid(ax, "y")
        if j == 1: ax.set_xlabel("age (y)")
        note(ax, f"minimum {q['vertex']:.1f} y", x=0.04, y=0.96)
        label(ax, let, x=-0.12)
    save(fig, "figS_cohorts")


figS_cohorts(D)

## 6 · Noise-free memory in exact arithmetic (SI Note 2.6, optional)

For symmetric $\tilde W = V\Lambda V^\top$ and $\eta\to0$ the input vector cancels and
$m(\tau) = \phi_\tau^\top K^{-1}\phi_\tau$, with $\phi_{\tau,k} = \lambda_k^\tau$ and
$K_{kl} = 1/(1-\lambda_k\lambda_l)$. $K$ is so ill-conditioned (~2000 bits needed) that double
precision is useless, so the solve is done in ball arithmetic (`pip install python-flint`) at two
precisions; their difference bounds the error. Writes `results/paper/data/noise_free_exact.csv`.

In [ ]:
def memory_curve_exact(lam, taus, bits):
    ctx.prec = bits
    L = [arb(repr(float(x))) for x in lam]
    n = len(L)
    K = arb_mat(n, n, [1 / (1 - L[k] * L[l]) for k in range(n) for l in range(n)])
    B = arb_mat(n, len(taus), [L[k] ** t for k in range(n) for t in taus])
    X = K.solve(B)
    out = []
    for j in range(len(taus)):
        s = arb(0)
        for k in range(n): s += B[k, j] * X[k, j]
        out.append(s)
    return out

In [ ]:
def run_exact(data, n=24, bits=(2500, 4000), out=DATA / "noise_free_exact.csv"):
    rho = 0.99   # N = 2 check: lambda = (rho, -rho) gives m(tau) = rho^(4 floor(tau/2)) (1 - rho^4)
    v = memory_curve_exact([rho, -rho], [0, 1, 2, 3], 256)
    print("N=2 check:", [round(float(x.mid()), 8) for x in v],
          "expected", [round(x, 8) for x in [1 - rho**4, 1 - rho**4, rho**4 * (1 - rho**4), rho**4 * (1 - rho**4)]])

    sid, mats, meta = ca.load_pickle(data) if str(data).endswith(".pkl") else ca.load_export(data)
    pick = np.linspace(0, len(sid) - 1, n).astype(int)
    taus = list(range(0, 21))
    rows, t0 = [], time.time()
    for i in pick:
        l = np.linalg.eigvalsh(mats[i]); l = rho * l / np.abs(l).max()
        lo, hi = memory_curve_exact(l, taus, bits[0]), memory_curve_exact(l, taus, bits[1])
        m = [float(x.mid()) for x in hi]
        rows.append(dict(sid=int(sid[i]), m0=m[0], MC_1_20=sum(m[1:]), min_m_1_20=min(m[1:]),
                         max_precision_diff=max(abs(float(a.mid()) - float(b.mid())) for a, b in zip(lo, hi)),
                         max_ball_radius=max(float(x.rad()) for x in hi)))
        print(f"  sid {sid[i]}: sum_(1..20) m = {rows[-1]['MC_1_20']:.12f}", flush=True)
    r = pd.DataFrame(rows)
    if "age" in meta: r = r.merge(meta[["sid", "age"]], on="sid", how="left")
    r.to_csv(out, index=False)
    print(f"{len(r)} connectomes in {time.time() - t0:.0f}s -> {out}")
    return r


if RUN_EXACT:
    from flint import arb, arb_mat, ctx
    run_exact(DATA_PKL)